# Compare the three sentence correctors
Run the cells from top to bottom (**Run All**). This notebook compares `solvers.py`, `solvers_spidy.py`, and `solvers_tanmay.py` on identical examples and saves predictions and scores separately.

Change the settings below. The default run takes about **10 minutes**: 100 examples × 2 seconds × 3 solvers, run sequentially. For a quick check, set `NUM_EXAMPLES = 3` and `TIMEOUT_SECONDS = 0.1`.

**Default dataset:** 100 distinct passages combining related corpus clauses and authored phrases on ten topics. Lengths vary from 90 to 129 words (557–904 characters); the generator targets 90–130 words per passage. Each passage keeps one topic.

Set `DATASET = "original"` to use the exact original 100 short test cases preserved in `data/original_100_test_cases.json`. Earlier results remain in their original folders.

Easy, medium, and hard correspond to 5%, 10%, and 20% character corruption. A 100-example run contains 34 easy, 33 medium, and 33 hard cases. Accuracy is measured against the original corpus text. These corpus-derived examples are not a held-out evaluation.

Requires a Python 3.7+ notebook kernel on macOS/Linux. The comparison code is included below; `compare_corpus_solvers.py` is not required.

In [26]:
from pathlib import Path
from datetime import datetime

NUM_EXAMPLES = 100
TIMEOUT_SECONDS = 2.0
SEED = 333

# "long": new contextual passages; "original": preserved first 100 short cases.
DATASET = "long"

# Fresh run: leave OUTPUT_FOLDER as None and RESUME as False.
# Resume: paste the previous run's folder here and set RESUME = True.
OUTPUT_FOLDER = None
RESUME = False
RETRY_ERRORS = True

# Normally detected automatically. Set an absolute path if needed.
PROJECT_FOLDER = None

if PROJECT_FOLDER is not None:
    PROJECT_DIR = Path(PROJECT_FOLDER).expanduser().resolve()
else:
    candidates = []
    for parent in (Path.cwd(), *Path.cwd().parents):
        candidates.extend([parent, parent / 'A1_my' / 'A1_fresh'])
    PROJECT_DIR = next((p for p in candidates if (p / 'solvers_spidy.py').is_file()
                        and (p / 'data' / 'corpus.txt').is_file()), None)
    if PROJECT_DIR is None:
        raise FileNotFoundError('Set PROJECT_FOLDER to the absolute path of A1_fresh.')

if RESUME and OUTPUT_FOLDER is None:
    raise ValueError('Set OUTPUT_FOLDER to the previous run folder when RESUME is True.')
RUN_DIR = (Path(OUTPUT_FOLDER).expanduser() if OUTPUT_FOLDER is not None else
           PROJECT_DIR / 'results' / ('comparison_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f')))
if not RUN_DIR.is_absolute():
    RUN_DIR = PROJECT_DIR / RUN_DIR
RUN_DIR = RUN_DIR.resolve()
print('Project:', PROJECT_DIR)
print('Results:', RUN_DIR)
print('Estimated search time: %.1f minutes' % (3 * NUM_EXAMPLES * TIMEOUT_SECONDS / 60))
if DATASET not in ("long", "original"):
    raise ValueError("DATASET must be long or original.")
EXAMPLES_FILE = PROJECT_DIR / "data" / ("long_corpus_examples/examples.json" if DATASET == "long" else "original_100_test_cases.json")
print("Dataset:", DATASET)


Project: /Users/aakritypandey/Documents/trying/ai/A1_my/A1_fresh
Results: /Users/aakritypandey/Documents/trying/ai/A1_my/A1_fresh/results/comparison_20261019_184727_002758
Estimated search time: 10.0 minutes
Dataset: long


## Comparison helpers
Run this cell once to load the benchmark and reporting functions. You do not need to edit it.

In [27]:
import argparse
import contextlib
import csv
import datetime
import hashlib
import importlib
import itertools
import json
import math
import os
import pickle
import random
import signal
import subprocess
import sys
import time
from pathlib import Path


ROOT = PROJECT_DIR
OUTPUT = RUN_DIR


class SearchDeadline(Exception):
    """Use a non-OSError deadline so filesystem helpers cannot swallow it."""


def export_case_comparison(output=None):
    """Export saved outputs and metrics together, one readable block per case.

    Reads saved results only; does not run or modify any solver.
    """
    output = Path(output) if output is not None else OUTPUT
    names = [('solvers_spidy', 'Spidy'), ('solvers_tanmay', 'Tanmay'), ('solvers', 'Yours')]
    saved = {}
    for name, _ in names:
        path = output / name / 'results.jsonl'
        if path.exists():
            rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
            if len({row['case_id'] for row in rows}) != len(rows):
                raise ValueError('Duplicate case IDs in %s' % path)
            saved[name] = {row['case_id']: row for row in rows}
    case_ids = sorted(set().union(*(set(rows) for rows in saved.values())))
    if not case_ids:
        raise ValueError('No saved cases found in %s' % output)

    def errors(expected, actual):
        return sum(a != b for a, b in itertools.zip_longest(expected, actual, fillvalue=None))

    def accuracy(expected, actual):
        return 100 * (1 - errors(expected, actual) / max(len(expected), len(actual), 1))

    lines = [
        'CASE-BY-CASE SOLVER COMPARISON',
        'Run folder: %s' % output.resolve(),
        'Cases: %d' % len(case_ids),
        'Actual = correct reference text. Yours = solvers.py.',
        'Word accuracy compares words at the same positions; character accuracy compares character positions.',
        'Accuracy and cost reduction: higher is better. Cost_fn = saved language-model cost; lower is better.',
        'Fixed errors count corrupted characters restored to the reference; introduced errors count previously correct positions made wrong.',
        'Timeouts score the saved best_state. Missing results and other failures are shown explicitly.', '',
    ]
    for case_id in case_ids:
        available = [rows[case_id] for rows in saved.values() if case_id in rows]
        reference = available[0]
        noisy, clean = reference['input'], reference['expected']
        if any(row['input'] != noisy or row['expected'] != clean or
               row['difficulty'] != reference['difficulty'] for row in available):
            raise ValueError('Case %s has mismatched inputs or references between solvers' % case_id)
        level = {'light': 'Easy', 'medium': 'Medium', 'heavy': 'Hard'}.get(
            reference['difficulty'], reference['difficulty'])
        heading = 'CASE %03d | %s | %d words | %d characters' % (case_id, level, len(clean.split()), len(clean))
        if reference.get('topic'):
            heading += ' | ' + reference['topic']
        lines.extend(['=' * 100, heading, '-' * 100,
                      'Input:  ' + noisy, 'Actual: ' + clean,
                      'Input metrics: word_accuracy=%.2f%% | character_accuracy=%.2f%% | character_errors=%d | cost_fn=%.6f' %
                      (accuracy(clean.split(), noisy.split()), accuracy(clean, noisy), errors(clean, noisy), reference['input_cost']),
                      'Actual metrics: word_accuracy=100.00%% | character_accuracy=100.00%% | character_errors=0 | cost_fn=%.6f' % reference['expected_cost'], ''])
        for name, label in names:
            row = saved.get(name, {}).get(case_id)
            if row is None:
                lines.extend([label + ' output: NOT AVAILABLE', ''])
                continue
            pred = row['prediction']
            remaining = errors(clean, pred)
            fixed = sum(a != b and p == a for a, b, p in itertools.zip_longest(clean, noisy, pred, fillvalue=None))
            introduced = sum(a == b and p != a for a, b, p in itertools.zip_longest(clean, noisy, pred, fillvalue=None))
            changed = errors(noisy, pred)
            lines.extend([
                label + ' output: ' + pred,
                '  Accuracy: word=%.2f%% | character=%.2f%% | exact_match=%s' %
                (accuracy(clean.split(), pred.split()), accuracy(clean, pred), str(pred == clean)),
                '  Errors: remaining=%d | fixed=%d | introduced=%d | changed_positions=%d' %
                (remaining, fixed, introduced, changed),
                '  Cost: cost_fn=%.6f | input_cost_reduction=%.6f | cost_minus_actual=%.6f' %
                (row['output_cost'], row['input_cost'] - row['output_cost'], row['output_cost'] - row['expected_cost']),
                '  Runtime: %.4fs | time_limit=%.4fs | timed_out=%s | error=%s' %
                (row['elapsed_seconds'], row['timeout_seconds'], str(row['timed_out']), row['error'] or 'None'), '',
            ])
    path = output / 'case_comparison.txt'
    path.write_text('\n'.join(lines) + '\n')
    return path


def summarize():
    lines = ['# Solver comparison', '',
             'All solvers use the same corpus examples and language model. '
             'Accuracy is measured against the original corpus sentences. '
             'These corpus-derived examples do not measure performance on unseen data.', '',
             '| Solver | Difficulty | Cases | Initial character errors | Remaining character errors | Word accuracy | Exact sentences | Timeouts | Failures |',
             '|---|---|---:|---:|---:|---:|---:|---:|---:|']
    all_rows = {}
    for name in ('solvers_spidy', 'solvers_tanmay', 'solvers'):
        path = OUTPUT / name / 'results.jsonl'
        if not path.exists():
            continue
        rows = [json.loads(line) for line in path.read_text().splitlines() if line]
        all_rows[name] = rows
        for level in ('light', 'medium', 'heavy', 'all'):
            group = [r for r in rows if level == 'all' or r['difficulty'] == level]
            if not group:
                continue
            lines.append('| %s | %s | %d | %d | %d | %.2f%% | %d | %d | %d |' % (
                name, level, len(group), sum(r['initial_errors'] for r in group),
                sum(r['remaining_errors'] for r in group),
                100 * sum(r['correct_words'] for r in group) / sum(r['words'] for r in group),
                sum(r['exact_match'] for r in group), sum(r['timed_out'] for r in group),
                sum(bool(r['error']) for r in group)))
    for name, rows in all_rows.items():
        if rows:
            lines.extend(['', '%s: %.2f seconds per example; %d completed examples.' %
                          (name, rows[0]['timeout_seconds'], len(rows)), ''])
    for first_name, second_name in itertools.combinations(all_rows, 2):
        first = {r['case_id']: r for r in all_rows[first_name]}
        second = {r['case_id']: r for r in all_rows[second_name]}
        pairs = [(first[i], second[i]) for i in sorted(first.keys() & second.keys())]
        wins_a = sum(a['remaining_errors'] < b['remaining_errors'] for a, b in pairs)
        wins_b = sum(b['remaining_errors'] < a['remaining_errors'] for a, b in pairs)
        lines.extend(['Pairwise character-error comparison across %d shared examples: '
                      '%s wins %d, %s wins %d, ties %d.' %
                      (len(pairs), first_name, wins_a, second_name, wins_b,
                       len(pairs) - wins_a - wins_b), ''])
    lines.extend(['Timeouts preserve each solver\'s best_state and are expected for solvers '
                  'that search indefinitely. Failures are reported separately. '
                  'Metrics include the saved output even when a case times out or fails.', ''])
    OUTPUT.mkdir(parents=True, exist_ok=True)
    (OUTPUT / 'summary.md').write_text('\n'.join(lines) + '\n')
    grouped = []
    labels = {'solvers': 'Yours', 'solvers_spidy': 'Spidy', 'solvers_tanmay': 'Tanmay'}
    for difficulty, label in [('light', 'Easy'), ('medium', 'Medium'), ('heavy', 'Hard')]:
        for name, rows in all_rows.items():
            group = [r for r in rows if r['difficulty'] == difficulty]
            if not group:
                continue
            grouped.append(dict(difficulty=label, solver=labels[name], cases=len(group),
                                word_accuracy_percent=100 * sum(r['correct_words'] for r in group) / sum(r['words'] for r in group),
                                exact_sentences=sum(r['exact_match'] for r in group),
                                remaining_character_errors=sum(r['remaining_errors'] for r in group),
                                mean_input_cost=sum(r['input_cost'] for r in group) / len(group),
                                mean_output_cost=sum(r['output_cost'] for r in group) / len(group)))
    if grouped:
        with (OUTPUT / 'difficulty_scores.csv').open('w', newline='') as target:
            writer = csv.DictWriter(target, fieldnames=list(grouped[0]))
            writer.writeheader()
            writer.writerows(grouped)
        table = ['# Scores by difficulty', '',
                 'Easy: 5% corruption; medium: 10%; hard: 20%. All solvers use the same examples and time limit.', '',
                 '| Difficulty | Solver | Cases | Word accuracy | Exact sentences | Remaining character errors | Average model cost |',
                 '|---|---|---:|---:|---:|---:|---:|']
        table.extend('| {difficulty} | {solver} | {cases} | {word_accuracy_percent:.2f}% | {exact_sentences}/{cases} | {remaining_character_errors} | {mean_output_cost:.3f} |'.format(**row) for row in grouped)
        table.extend(['', 'Higher accuracy is better; lower model cost is better. Compare model costs within each difficulty group.'])
        (OUTPUT / 'difficulty_scores.md').write_text('\n'.join(table) + '\n')
    if all_rows:
        print('Case-by-case outputs and metrics: %s' % export_case_comparison())
    print('\n'.join(lines))


def run(args):
    cases = json.loads(args.examples.read_text())
    matrix = json.loads((ROOT / 'data/conf_matrix.json').read_text())
    solver_class = importlib.import_module(args.solver).SentenceCorrector
    with (ROOT / 'data/lm_model.pkl').open('rb') as source:
        model = pickle.load(source)
    model.unk_prob = 1e-20
    model.set_mode('spell_check')
    folder = OUTPUT / args.solver
    folder.mkdir(parents=True, exist_ok=True)
    if args.limit > len(cases):
        raise ValueError('Requested %d examples, but only %d exist.' % (args.limit, len(cases)))
    manifest = dict(solver=args.solver, timeout=args.timeout, seed=args.seed, cases=cases,
                    solver_sha256=hashlib.sha256((ROOT / (args.solver + '.py')).read_bytes()).hexdigest(),
                    confusion_matrix=matrix)
    manifest_path = folder / 'manifest.json'
    if manifest_path.exists():
        if not args.resume:
            raise ValueError('This output folder already contains a run. Use --resume or choose a new --output.')
        if json.loads(manifest_path.read_text()) != manifest:
            raise ValueError('Solver code, settings, or examples changed; choose a new --output folder.')
    else:
        manifest_path.write_text(json.dumps(manifest, indent=2) + '\n')
    cases = cases[:args.limit]
    (folder / 'input.txt').write_text(''.join(case['input'] + '\n' for case in cases))
    (folder / 'expected.txt').write_text(''.join(case['original'] + '\n' for case in cases))
    result_path = folder / 'results.jsonl'
    rows = [json.loads(line) for line in result_path.read_text().splitlines() if line] if result_path.exists() else []
    assert [r['case_id'] for r in rows] == list(range(1, len(rows) + 1))
    if len(rows) > len(cases):
        raise ValueError('Saved results exceed the requested limit; use a fresh output folder.')

    def expire(signum, frame):
        raise SearchDeadline('Per-case time limit reached')

    previous = signal.signal(signal.SIGALRM, expire)
    try:
        with open(os.devnull, 'w') as quiet:
            indices = ([index for index, row in enumerate(rows) if row['error']]
                       if args.retry_errors else []) + list(range(len(rows), len(cases)))
            for index in indices:
                case = cases[index]
                random.seed(args.seed + index)
                solver = solver_class(model, matrix)
                timed_out, error = False, ''
                started = time.monotonic()
                try:
                    with contextlib.redirect_stdout(quiet):
                        signal.setitimer(signal.ITIMER_REAL, args.timeout)
                        if args.solver == 'solvers':
                            solver.search(case['input'], result_path=folder / 'passes.txt')
                        else:
                            solver.search(case['input'])
                except SearchDeadline:
                    timed_out = True
                except Exception as exc:
                    error = '%s: %s' % (type(exc).__name__, exc)
                finally:
                    signal.setitimer(signal.ITIMER_REAL, 0)
                elapsed = time.monotonic() - started
                pred = solver.best_state
                if not isinstance(pred, str):
                    error = error or 'Solver did not save a string best_state'
                    pred = case['input']
                clean = case['original']
                # The assignment's operators preserve length; flag violations.
                if len(pred) != len(clean):
                    error = error or 'Output length differs from original'
                remaining = sum(a != b for a, b in zip(clean, pred)) + abs(len(clean) - len(pred))
                words, predicted_words = clean.split(), pred.split()
                row = dict(case_id=index + 1, corpus_line=case['corpus_line'],
                           topic=case.get('topic', ''),
                           difficulty=case['difficulty'], input=case['input'], expected=clean,
                           prediction=pred, initial_errors=case['substitutions'],
                           remaining_errors=remaining, words=len(words),
                           correct_words=sum(a == b for a, b in zip(words, predicted_words)),
                           exact_match=pred == clean, input_cost=model(case['input']),
                           output_cost=model(pred), expected_cost=model(clean),
                           elapsed_seconds=elapsed, timeout_seconds=args.timeout,
                           timed_out=timed_out, error=error)
                if index < len(rows):
                    rows[index] = row
                else:
                    rows.append(row)
                checkpoint = result_path.with_suffix('.tmp')
                checkpoint.write_text(''.join(json.dumps(saved) + '\n' for saved in rows))
                checkpoint.replace(result_path)
                if (index + 1) % 25 == 0 or error or index == len(cases) - 1:
                    print('%s: %d/%d completed; latest errors %d -> %d%s' %
                          (args.solver, index + 1, len(cases), case['substitutions'], remaining,
                           ('; ' + error) if error else ''), flush=True)
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, previous)
    (folder / 'predictions.txt').write_text(''.join(r['prediction'] + '\n' for r in rows))
    fields = [key for key in rows[0] if key not in ('input', 'expected', 'prediction')]
    with (folder / 'metrics.csv').open('w', newline='') as target:
        writer = csv.DictWriter(target, fieldnames=fields, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)
    print('%s finished: %s' % (args.solver, folder), flush=True)




In [28]:
from types import SimpleNamespace
from IPython.display import Markdown, HTML, display
import html

if not hasattr(signal, 'setitimer'):
    raise RuntimeError('This notebook requires macOS or Linux for per-example timeouts.')
if not isinstance(NUM_EXAMPLES, int) or isinstance(NUM_EXAMPLES, bool) or NUM_EXAMPLES < 1:
    raise ValueError('NUM_EXAMPLES must be a positive integer.')
if not math.isfinite(TIMEOUT_SECONDS) or TIMEOUT_SECONDS <= 0:
    raise ValueError('TIMEOUT_SECONDS must be finite and positive.')

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
examples_path = EXAMPLES_FILE
if not examples_path.exists():
    if DATASET == 'long':
        subprocess.run([sys.executable, str(PROJECT_DIR / 'generate_long_examples.py'),
                        '--seed', str(SEED)], check=True)
    else:
        raise FileNotFoundError('The preserved original cases file is missing: %s' % examples_path)
examples = json.loads(examples_path.read_text())
if NUM_EXAMPLES > len(examples):
    raise ValueError('Only %d examples are available.' % len(examples))
selected_examples = examples[:NUM_EXAMPLES]
for level, label in [('light', 'Easy'), ('medium', 'Medium'), ('heavy', 'Hard')]:
    print('%s: %d examples' % (label, sum(c['difficulty'] == level for c in selected_examples)))

print('Selected passage lengths: %d–%d words' % (
    min(len(c['original'].split()) for c in selected_examples),
    max(len(c['original'].split()) for c in selected_examples)))


def run_solver(name):
    # Reload edits made since a previous execution in this kernel.
    importlib.invalidate_caches()
    if name in sys.modules:
        importlib.reload(sys.modules[name])
    args = SimpleNamespace(solver=name, timeout=TIMEOUT_SECONDS, seed=SEED,
                           limit=NUM_EXAMPLES, resume=RESUME, retry_errors=RETRY_ERRORS,
                           examples=EXAMPLES_FILE)
    print('Running %s...' % name, flush=True)
    run(args)

Easy: 34 examples
Medium: 33 examples
Hard: 33 examples
Selected passage lengths: 90–129 words


## Your solver

In [29]:
run_solver('solvers')

Running solvers...
solvers: 25/100 completed; latest errors 35 -> 0
solvers: 50/100 completed; latest errors 69 -> 21
solvers: 75/100 completed; latest errors 137 -> 64
solvers: 100/100 completed; latest errors 37 -> 4
solvers finished: /Users/aakritypandey/Documents/trying/ai/A1_my/A1_fresh/results/comparison_20261019_184727_002758/solvers


## Spidy

In [30]:
run_solver('solvers_spidy')

Running solvers_spidy...
solvers_spidy: 25/100 completed; latest errors 35 -> 4
solvers_spidy: 50/100 completed; latest errors 69 -> 27
solvers_spidy: 75/100 completed; latest errors 137 -> 72
solvers_spidy: 100/100 completed; latest errors 37 -> 10
solvers_spidy finished: /Users/aakritypandey/Documents/trying/ai/A1_my/A1_fresh/results/comparison_20261019_184727_002758/solvers_spidy


## Tanmay

In [31]:
run_solver('solvers_tanmay')

Running solvers_tanmay...
solvers_tanmay: 25/100 completed; latest errors 35 -> 35
solvers_tanmay: 50/100 completed; latest errors 69 -> 69
solvers_tanmay: 75/100 completed; latest errors 137 -> 137
solvers_tanmay: 100/100 completed; latest errors 37 -> 37
solvers_tanmay finished: /Users/aakritypandey/Documents/trying/ai/A1_my/A1_fresh/results/comparison_20261019_184727_002758/solvers_tanmay


## Easy, medium, and hard results
Higher word accuracy and more exact sentence matches are better. Lower language-model cost is better; compare costs within the same difficulty group. A timeout is expected for a solver that searches indefinitely: its saved `best_state` is scored. Other errors appear in `summary.md` and the per-case files.

`case_comparison.txt` puts each input, actual reference, Spidy output, Tanmay output, and your output together with accuracy, cost, character errors, runtime, and timeout status. It is generated automatically from saved results.


In [32]:
import io
# Rebuild saved Markdown and CSV reports, then display the grouped table.
with contextlib.redirect_stdout(io.StringIO()):
    summarize()
display(Markdown((RUN_DIR / 'difficulty_scores.md').read_text()))
print('Saved reports:', RUN_DIR)
print('Overall results: summary.md')
print('Per solver: predictions.txt, metrics.csv, and results.jsonl')
print('Detailed comparison:', RUN_DIR / 'case_comparison.txt')


# Scores by difficulty

Easy: 5% corruption; medium: 10%; hard: 20%. All solvers use the same examples and time limit.

| Difficulty | Solver | Cases | Word accuracy | Exact sentences | Remaining character errors | Average model cost |
|---|---|---:|---:|---:|---:|---:|
| Easy | Spidy | 34 | 97.21% | 2/34 | 197 | 191.359 |
| Easy | Tanmay | 34 | 75.46% | 0/34 | 1071 | 664.176 |
| Easy | Yours | 34 | 98.63% | 11/34 | 93 | 165.959 |
| Medium | Spidy | 33 | 91.18% | 0/33 | 731 | 323.395 |
| Medium | Tanmay | 33 | 57.09% | 0/33 | 2065 | 1029.228 |
| Medium | Yours | 33 | 93.12% | 1/33 | 566 | 277.568 |
| Hard | Spidy | 33 | 72.40% | 0/33 | 2496 | 665.718 |
| Hard | Tanmay | 33 | 31.50% | 0/33 | 4134 | 1465.347 |
| Hard | Yours | 33 | 72.59% | 0/33 | 2469 | 649.730 |

Higher accuracy is better; lower model cost is better. Compare model costs within each difficulty group.


Saved reports: /Users/aakritypandey/Documents/trying/ai/A1_my/A1_fresh/results/comparison_20261019_184727_002758
Overall results: summary.md
Per solver: predictions.txt, metrics.csv, and results.jsonl
Detailed comparison: /Users/aakritypandey/Documents/trying/ai/A1_my/A1_fresh/results/comparison_20261019_184727_002758/case_comparison.txt


## Inspect an individual example
Change `CASE_ID` below to compare the original sentence with all three predictions.

In [33]:
CASE_ID = 1  # 1 through NUM_EXAMPLES
if not 1 <= CASE_ID <= NUM_EXAMPLES:
    raise ValueError('CASE_ID must be between 1 and NUM_EXAMPLES.')
example = selected_examples[CASE_ID - 1]
print('Difficulty:', {'light': 'Easy', 'medium': 'Medium', 'heavy': 'Hard'}[example['difficulty']])
print('Input:   ', example['input'])
print('Expected:', example['original'])
for name in ('solvers', 'solvers_spidy', 'solvers_tanmay'):
    records = [json.loads(line) for line in (RUN_DIR / name / 'results.jsonl').read_text().splitlines()]
    result = next(row for row in records if row['case_id'] == CASE_ID)
    print('\n%s: %s' % (name, result['prediction']))
    print('  Remaining character errors: %d; model cost: %.3f%s' %
          (result['remaining_errors'], result['output_cost'],
           '; error: ' + result['error'] if result['error'] else ''))

Difficulty: Easy
Input:    central banks reviewed conditions in the fareign exchange parket the recent stabilisy would last umtil marketd decided to test the resolve of central banks he said under such circumstances the countries agreed thair currencies were within ranges broadly consestent with economig fundamintals poohl sayz carrercy market intervedtion cannat substidute for correct economic policies if exchange retes are idbaranced analysts said the yen also gained as traders unwiund long sterling stoet mark positions ustablishad lately officials said monetary tolicy would remain importand in the coming weeks the report said more details would be availadle noxt month
Expected: central banks reviewed conditions in the foreign exchange market the recent stability would last until markets decided to test the resolve of central banks he said under such circumstances the countries agreed their currencies were within ranges broadly consistent with economic fundamentals poehl says currenc

Completed cases are checkpointed after each example. If you interrupt the notebook, set `OUTPUT_FOLDER` to the printed results folder and `RESUME = True`, then run the notebook again with the same example count, timeout, seed, and solver code. After editing a solver, start a fresh run with `OUTPUT_FOLDER = None` and `RESUME = False`.